## From text data to data batches

This code in this notebook follows the presentation in Chapter 3 of

Raschka, Sebastion. 2025. *Build a Large Language Model (from Scratch)*.

made available on our Canvas through our library. Code has been adapted in various places (mostly to compress the presentation, so consult the original for a more leisurely presentation).

### Load the data

In [2]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
# to save as a file locally
#urllib.request.urlretrieve(url, file_path)

In [3]:
with urllib.request.urlopen(url) as fh:
    wharton_text = fh.read().decode(encoding="UTF8")

This is untokenized text, so this is string length, not number of words.  So: we have very little data in  language modeling terms.

In [5]:
len(wharton_text)

20479

### Tokenizing

The regular expression below defines **word separators**, characters or sequences of characters that occur **between** words. Everywhere there's a match, we split the string, leaving out the separators.

This is **not** the approach to tokenizing used by the GPT-2 tokenizer we'll introduce below, which seeks to discover words by empirical criteria rather than by orthographic convention (spaces separate words).  But it's
a good place to start.

In [4]:
import re
english_tok_re = r'([,.:;?_!"()\']|--|\s)'
preprocessed = re.split(english_tok_re, wharton_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


That is approximately the number of word tokens we'll end up with.

In [7]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

###  Vocab & preprocessing tokenizer

Take the tokenized text and assign an integer index to each distinct word:

In [6]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)
vocab = {token:integer for integer,token in enumerate(all_words)}

1130


In [7]:
import nltk
from nltk import FreqDist
fd = FreqDist(preprocessed)

In [8]:
len(fd)

1130

Language data sets often show this property:  More than half of the vocabulary consists of words that occur only once in the text.

In [9]:
hapax = [wd for (wd,ct) in fd.most_common() if ct ==1]
print(len(hapax)/len(fd))

hapax[:20]

0.6168141592920354


['HAD',
 'cheap',
 'genius',
 'widow',
 'established',
 'villa',
 'Though',
 'Rome',
 'Florence',
 'Gideon',
 'Chicago',
 'sitter',
 'deploring',
 'unaccountable',
 'going',
 'send',
 'value',
 'loss',
 'Arrt',
 'multiplied']

We assign integer indexes to each word to facilitate efficient processing down the line:

In [10]:
vocab = {token:integer for integer,token in enumerate(all_words)}

class SimpleTokenizerV2:
    
    english_tok_re = r'([,.:;?_!"()\']|--|\s)'
    
    def __init__(self, vocab):
        self.str_to_int = vocab 
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(self.english_tok_re, text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int            #1
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)    #2
        return text

In [11]:
tok2 = SimpleTokenizerV2(vocab)
tok2_ids = tok2.encode(wharton_text)
len(tok2_ids)

4690

In [12]:
print(tok2_ids[:20])

[53, 44, 149, 1003, 57, 38, 818, 115, 256, 486, 6, 1002, 115, 500, 435, 392, 6, 908, 585, 1077]


In [13]:
text23 = tok2.decode(tok2_ids[:20])
text23

'I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow enough -- so it was'

Here's our current vocab size and the number of tokens::

In [14]:
len(set(tok2_ids)), len(tok2_ids)

(1130, 4690)

### THe GPT-2 tokenizer

In [15]:
# ! conda install tiktoken

In [34]:
# Includes the GPT-2 tokenizer
import tiktoken
from importlib.metadata import version
print("tiktoken version:", version("tiktoken"))
# the BPE tokenizer
gpt_tokenizer = tiktoken.get_encoding("gpt2")

tiktoken version: 0.12.0


In [35]:
# the BPE tokenizer
gpt_tokenizer = tiktoken.get_encoding("gpt2")

Maximum encoding value used:

In [14]:
gpt_tokenizer.max_token_value

50256

Num vocab consistent with continuous encoding starting at 0 (Whew!):

In [15]:
gpt_tokenizer.n_vocab

50257

The complete stored vocab:

In [19]:
vocab = [gpt_tokenizer.decode_single_token_bytes(cd) for cd in range(gpt_tokenizer.n_vocab)]
print(len(vocab))

50257


The byte-pair encoding algorithm does not rule out words with spaces in them. Let's see if we got any:

In [20]:
# To look for words that have spaces IN them (not first character)
# try unicode  char 160: the no-break space. Also regular space (32) 
LL =[wd for wd in vocab if len(wd)>1 and 160 in wd[1:] or 32 in wd[1:]]
print(len(LL))

20


This shows there are some very odd birds indeed in our "vocabulary".

In [22]:
for (i, x) in enumerate(LL):
    try:
      print(f"{i+1}: |{x.decode()}|")
    except UnicodeDecodeError:
      print(f"{i+1}: Error on ", x)

1: | |
2: |  |
3: |  |
4: |    |
5: |    |
6: |        |
7: |        |
8: |à|
9: |ム|
10: | à|
11: |■|
12: |   |
13: |†|
14: | ■|
15: |                |
16: Error on  b'\xe6\xa0'
17: |
 |
18: |                |
19: |だ|
20: |   |


Looks like item 17 has a line break in it.

In [24]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     " of someunknownPlace."
)
integers = gpt_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
## This is indeed just what it looks like, a list of integers
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [25]:
strings = gpt_tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


### Unknown words

A major motivation for using the GPT tolenizer is its handling of unknown words.

In [26]:
text2 = ("quegliamar is great" )
enc2 = gpt_tokenizer.encode(text2, allowed_special={"<|endoftext|>"})

In [27]:
enc2

[421, 1533, 5058, 283, 318, 1049]

In [28]:
gpt_tokenizer.decode(enc2)

'quegliamar is great'

In [29]:
[gpt_tokenizer.decode([idx]) for idx in enc2]

['qu', 'eg', 'liam', 'ar', ' is', ' great']

The letter chunks are merged back together during ordinary decoding:

In [30]:
gpt_tokenizer.decode(enc2)

'quegliamar is great'

The algorithm used here is called **byte-pair encoding tokenization**.  Basically, the vocabulary starts as all the characters found in the data (and no words!).  Then the most frequently occurring pair of vocabulary items are
merged into a new vocabulary item and the counts are recomputed. Then we merge again, and so on, until all remaining pairs occur fewer times than some threshhold.  You can learn more about the byte pair encoding tokenization [here.](https://huggingface.co/learn/llm-course/en/chapter6/5)

### GPT-2 on wharton

Take one:

In [31]:
wharton_ids_gpt2 = gpt_tokenizer.encode(wharton_text)
wharton_wds_gpt2 = gpt_tokenizer.decode(wharton_ids_gpt2)
wharton_vocab_gpt2 = sorted(set(wharton_wds_gpt2.split()))

Have a look at what it does.  Note what it does to the unknown word created by the main character's last name.

In [33]:
wharton_wds_gpt2 = gpt_tokenizer.decode_tokens_bytes(wharton_ids_gpt2)
#wharton_wds_gpt2 = tokenizer.decode_with_offsets(wharton_ids_gpt2)

In [34]:
wharton_wds_gpt2[5:9]

[b' Jack', b' G', b'is', b'burn']

Two things going on here.  

1. We're getting byte strings, not strings, which means that to find the corresponding string, we need to know (a) that it's text data (byte strings can represent any kind of data); and (b) what encoding (like utf-8 or utf-16) the data is represented in.   Using byte strings means we can deal equally well with Roman characters, rare writing sets represented as 2 bytes in the unicode, and, of course emoji.  This output **looks** like English characters because we are dealing with byte values < 256.  This is called **byte level** tokenization.
2. Byte pair encoding is splitting up very rare words because they don't meet the frequency threshhold for being included in the vocabulary.

The unicode symbol  with hexadecimal code 394 (decimal 916 ), GREEK LETTER CAPITAL DELTA:

In [35]:
"\u0394",f"{916:x}"

('Δ', '394')

A multi-byte character (a Mahjong tile)

In [36]:
"\U0001F01A"

'🀚'

Another (an emoji):

In [37]:
"\U0001F62D"

'😭'

Because of exploding unknown words into character chunks, the number of "word" tokens went up from 4,690 to 5,145.

In [38]:
len(wharton_ids_gpt2)

5145

The vocab size also went up:

In [41]:
len(set(wharton_ids_gpt2)),len(wharton_vocab_gpt2)

(1416, 1485)

##  Torch batch processor

This is more appropriate for the official training loop than the model components but we'll
foolow Raschke and introduce batch processing tools here.

We process more efficiently by passing **batches** of data through the learner. Also, more importantly,
since we update the weights (by back propagation) after each batch is processed,  using larger batchsizes
reduces noise in the weight updates.

In [24]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    
    def __init__(self, txt, tokenizer, max_length, stride):
        """
        {input} and {target} are word sequences of the same length,
        identical except that they are offset by 1 words.  For example:
        
        input: saw the cat on the mat
        target: the dat on the ma6 today
        
        The idea is that as we scan the input word by word we will
        try to predict the target in parallel.   We call
        the length of input and target {max_length} (because
        at the end of document the length of {input} may be less
        than {max_length}).
        
        Initialize a dataset with datasource `txt`.  `self.input_ids`
        contains the dataset  tokenized  {token_ids} and chunked
        into {max_length} sized chunks, with the starting positions
        of the chunks {stride} apart; {self.target_ids} contains
        a copy of each chunk offset by 1.  If stride is 1, the ith chunk 
        of {input_ids} contains token_ids[i:i+max_length]
        and the ith chunk of target_ids contains token_ids[i+1:i+max_length+1]
        
        This creates a corpus suitable for the supervised learning
        of a sequence to sequence mapping.  Self.input_ids
        contains the input sequence; and due to the
        the offsetof 1, the target consumes 1 more word from the data.
        So what the learner is going to learner is how to predict the next 
        word based on a history of size {max_length}.
        """
        self.input_ids = []
        self.target_ids = []
        self.tokenizer = tokenizer

        token_ids = self.tokenizer.encode(txt)    #1

        for i in range(0, len(token_ids) - max_length, stride):     #2
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):    #3
        return len(self.input_ids)

    def __getitem__(self, idx):         #4
        return self.input_ids[idx], self.target_ids[idx]

In [18]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0,
                         make_local_vocab=False):
    tokenizer = tiktoken.get_encoding("gpt2")   #1
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)   #2
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,     #3
        num_workers=num_workers     #4
    )

    return dataloader

###  Dataloader example

In [44]:
#with open("the-verdict.txt", "r", encoding="utf-8") as f:
#    raw_text = f.read()


dataloader = create_dataloader_v1(
    wharton_text, batch_size=1, max_length=4, stride=1, shuffle=False,
    make_local_vocab=True)
# we can loop through this to get the input,target pairs from the corpus.
#data_iter = iter(dataloader)      #1
#first_batch = next(data_iter)
#print(first_batch)

for (i,batch) in enumerate(dataloader):
    if i == 3:
        break
    else:
        print(batch[0].shape)
        print(batch)
        print()

torch.Size([1, 4])
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

torch.Size([1, 4])
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]

torch.Size([1, 4])
[tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]



### Data loading with larger batches

In [19]:
batch_size,max_length, stride = 8,4,4

dataloader = create_dataloader_v1(
    wharton_text, batch_size=batch_size, max_length=max_length, stride=stride,
    shuffle=False,make_local_vocab=True
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs: {inputs.shape} (batch size x sequence length) \n", inputs)
print("\nTargets:\n", targets)

Inputs: torch.Size([8, 4]) (batch size x sequence length) 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Embeddings

The word embeddings of the model can be thought of as the initial layer of a neural net.  Many layers in
a torch neural net have a `weights` attribute where the learned weights are stored and updated.
the is where the floating point values of our word embeddings model can be found.

The embedding vaues will be learned, so We start with randomly initialized embeddings.

In [20]:
torch.manual_seed(123)
embedding_dim = 200
#vocab_size computed above
tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
print(f"{vocab_size=} {embedding_dim=}")
embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
print(embedding_layer.weight.shape)

vocab_size=50257 embedding_dim=200
torch.Size([50257, 200])


Look up embeddings for the first input sequence in  `inputs`:

In [21]:
inputs[0]

tensor([  40,  367, 2885, 1464])

Each input vector of 4 word idxs (representing our history-size) gets mapped to a 4 x 200 
tensor, where 200 is the number of dimensions in our embeddings.

In [22]:
embedding_layer(inputs[0]).shape

torch.Size([4, 200])

The entire dataset gets mapped to a `batch_size x sequence length x embedding_dims`:

In [23]:
print(inputs.shape)
token_embeddings = embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4])


torch.Size([8, 4, 200])

This is the shape of the input to training.

## Positional embeddings

In [882]:
torch.arange(context_length)

tensor([0, 1, 2, 3, 4, 5])

In [883]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, embedding_dim)
#print(pos_embedding_layer.weight[:,:20])
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
#pos_embeddings = pos_embedding_layer(torch.arange(2))
#print(pos_embeddings[:,:20])
pos_embedding_layer

Embedding(4, 200)

Call the  tensor whose first 20 elements are printed below `pos_embeddings` row 1,
and call the next row row 2.

In [884]:
pos_embeddings[0,:20]

tensor([ 0.0715, -0.1139,  0.1233, -1.1925,  1.0893, -0.3283, -0.0329, -0.4701,
         0.2451, -0.9716,  1.0316, -0.2396, -0.7550, -0.1044,  0.0980,  1.2820,
        -0.6905,  1.8489,  0.2027,  0.1932], grad_fn=<SliceBackward0>)

Positional encodings are added to the `token_embeddings`  to give `input_embeddings`:

In [885]:
input_embeddings = token_embeddings + pos_embeddings

The `input_embeddings` tensor is the same shape as `token_embeddings`:  

In [886]:
input_embeddings.shape

torch.Size([8, 4, 200])

The `pos_embeddings` tensor is
added to each of the 8  batches (broadcasting) in `token_embeddings`.
Thus, each tensor that is first in a batch sequence gets `pos_embeddings` row 1  added to it; each that is second
gets `pos_embeddings` row 2  added, and so on.

##  Attention:  First pass

Raschka's simplified example

> The dot product is a measure of similarity because it quantifies how closely two vectors are aligned: a higher dot product indicates a greater degree of alignment or similarity between the vectors.  --Raschke (3.4)

#### Defining the inputs (used for the next few sections)

The key attention computation in the simplified model:  How similar 
are each of the inputs in the current history to the query vector?

In [28]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# The current word
query = inputs[1]  
# softmax layer after the (linear) "similarity" mapping.
sim1 = inputs@query
attn_weights_2 = torch.softmax(sim1,dim=0)
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

There are 6 attention weights because there are 6 inputs we are weighting for importance wrt to
the query vector `inputs[1]`.  Note that the current word only has attention weight .2379.

The weights have been normalized by the softmax step:

In [29]:
attn_weights_2.sum()

tensor(1.)

#### Computing the context vector (applying the weights to the inputs and summing)  

In [30]:
context_vec_2 = attn_weights_2@inputs
context_vec_2

tensor([0.4419, 0.6515, 0.5683])

Note that `context_vec_2[i]` is `attn_weights_2.dot(inputs[:,i])`.

In [31]:
i=1
context_vec_2[i], inputs[:,i].dot(attn_weights_2)

(tensor(0.6515), tensor(0.6515))

That is, **the second component of the context vector is the weighted sum of the second components
of all the inputs.**  Later we will transform the input before computing this weighted sum.  But the idea
is still the same.

#### Takeaway

The `context_vec_2`
can be viewed as a version of the query vector transformed by its weighted context;
or simply as a representation of the query vector  the current context words

#### Computing all the context vectors all the inputs simultaneously

Computing the similarity scores.  This works because the similarity score is computed by taking the **dot product** of two vectors:

In [32]:
sim_M = inputs@inputs.T
# sim_M[i,j] is the similarity of the inputs[i] vector
# to the inputs[j] vector
sim_M

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

Note the scores are are **not** normalized (if they were, we would have 
rows that add to 1).  Here is a row-normalized version,

Computing the similarity scores and normalizing:

In [33]:
attn_weights = torch.softmax(inputs@inputs.T,dim=-1)

In [34]:
attn_weights.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

To compute the context vectors, just apply the attention weights to the input vectors.

In [35]:
context_vecs = attn_weights @ inputs
context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

Note that `context_vecs[1,:]` is the same same context vector we computed
using `inputs[1]` as our query vector.

##  Rendering the pipeline in pytorch

We'll put together the pieces discussed above as a collection of matrix multiplications (and a little
softmaxing).

So far: attention weights have all been determined by one relation between word vectors -- **similarity**
-- defined by the dot product between two vectors.

The problem is that any of the input vector has to serve all these different functions:

1.  Characterize the word when it is the current word whose attention weights we are computing **query function**
2.  Characterize the  word when it is in the context of a word whose attention weights are being computed.  **key function**.
3.  Characterize the word when it is combined with the attention weights to create the final **context vector**.


What we will do in the more realistic attention module described in this section is transform the input vector into
three different vectors, one for each of the three functions

```python
key = inputs[1] @ W_key 
query = inputs[1] @ W_query
value = inputs[1] @ W_value
```

`W_key`, `W_query`, and `W_value` are transformations defined by training.  Then

1.  Combining the keys with the queries produces the attention scores for the current sequence.
2.  Softmaxing and scaling produces the attention weights.
3.  Combining the attention weight with the value vectors produces the context vector, the output
     of the attention module.

We present the code in `torch` and introduce some of the more important features of `torch`
along the way.

A parameter is a value in a neural network that will be optimized during training. 
That is, these are learned matrices which will have `autograd`
turned on, to allow back propagation. As above, they are be used to transform inputs
into query, key, and value vectors respectively.  

In [36]:
x_2 = inputs[1]     #1
d_in = inputs.shape[1]      #2
d_out = 2         #3
d_k = d_out

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_k), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_k), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Compute the query, key and value vectors
#query_2 = x_2 @ W_query 
keys = inputs @ W_key 
values = inputs @ W_value
queries = inputs @ W_query
#query_2 = x_2 @ W_query 
query_2 = queries[1]
print(query_2)

tensor([0.4306, 1.4551])


In [37]:
x_2 @ W_query 

tensor([0.4306, 1.4551])

### Matrix for the weighted attention scores (batch size by sequence length)

In [38]:
# or keys@queries.T (to get the attentions scores in columns)
attn_scores = queries@keys.T
attn_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

Note that `attention_scores[1,:]` is:

In [39]:
#keys.shape 6x2
keys@query_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

Next the `attention_weights` matrix is computed using softmax and scaling:

In [40]:
scaled_dps = attn_scores / (d_k**0.5)
attn_weights = torch.softmax(scaled_dps,dim=-1)
attn_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

Continuing to track `inputs[1]`:  these are the attention weights wrt to `inputs[1]`.

In [41]:
attn_weights[1]

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

To get the context vecs we combine the attn_weights with the value vectors:

In [42]:
context_vecs = attn_weights@values
context_vecs

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])

So the context vector for `inputs[1]`  is

In [43]:
context_vecs[1]

tensor([0.3061, 0.8210])

##  An attention module

We package the operations we've been performing into
**torch** class.  The code below looks a little more like a functioning
pytorch `nn`.  It is just a first attempt.

The `__init__` method initializes our key, query and value matrices $W_{K}$, $W_{Q}$, and $W_{V}$,
as above.  The `forward` method defines what will happen tot he input `x` and returns an ouput,
in this case, the context_vec we computed above.

In [44]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key  = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec


Using it:

```
d_in context_length (sequence length)
d_out context vector dimension (need not be the same as embedding dimensionality)
```

Practically speaking 

In [45]:
print(inputs.shape)
d_in = inputs.shape[1]      #2
d_out = 2         #3
torch.manual_seed(123)
lention_v1(d_in, d_out)
# Apply the learner to the input and produce context vectors.
outputs = sa_v1(inputs)
print(outputs.shape)
print(outputs)

torch.Size([6, 3])
torch.Size([6, 2])
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Another step toward torchification:  Implement the weight matrices and their straightforward matrix multiplications with `nn.Linear` layers.  Proper initialization of NN weights is a subject some thought has been given to.
The `nn.Linear` layer will do the matrix multiplications we want but will also initialize the weights
in a more learning-compatible way.

In [47]:
class SelfAttention_v2(nn.Module):
    
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

Same way of using it:

In [48]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


EXERCISE 3.1 (from Raschka) COMPARING SELFATTENTION_V1 AND SELFATTENTION_V2
Note that nn.Linear in SelfAttention_v2 uses a different weight initialization scheme as `nn.Parameter(torch.rand(d_in, d_out))` used in SelfAttention_v1, which causes both mechanisms to produce different results. To check that both implementations, SelfAttention_v1 and SelfAttention_v2, are otherwise similar, we can **transfer the weight matrices from a SelfAttention_v2 object to a SelfAttention_v1**, such that both objects then produce the same results.

Your task is to correctly assign the weights from an instance of SelfAttention_v2 to an instance of SelfAttention_v1. To do this, you need to understand the relationship between the weights in both versions. (Hint: nn.Linear stores the weight matrix in a transposed form.) After the assignment, you should observe that both instances produce the same outputs.

Demonstrating that in their intial states they are different NNs:

In [97]:
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.6923, 1.1926],
        [0.7188, 1.2361],
        [0.7183, 1.2352],
        [0.6891, 1.1882],
        [0.6897, 1.1880],
        [0.6968, 1.2012]], grad_fn=<MmBackward0>)


In [96]:
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[ 0.3671, -0.3086],
        [ 0.3675, -0.3095],
        [ 0.3675, -0.3094],
        [ 0.3670, -0.3073],
        [ 0.3670, -0.3061],
        [ 0.3672, -0.3085]], grad_fn=<MmBackward0>)


Transferring the weight matrices from a `SelfAttention_v2` instance to a `SelfAttention_v1` instance.

In [88]:
sa_v1.W_query.data = sa_v2.W_query.weight.T

In [89]:
sa_v1.W_key.data = sa_v2.W_key.weight.T

In [90]:
sa_v1.W_value.data = sa_v2.W_value.weight.T

You also need to up  date the structure that keeps bundles together all your parameter info, for purposes of saving and loading models, the **state_dict**.

In [91]:
sa_v1.state_dict()['W_query'] = sa_v2.state_dict()['W_query.weight'].T

Read more about state dicts in the [torch state_dict tutorial.](https://docs.pytorch.org/tutorials/recipes/recipes/what_is_state_dict.html)

Demonstrating they are now the same NNs:

In [92]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [93]:
print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Causal attention

Problem. Once we have formulated the attention module as a sequence of linear transformations on
all the words in the current context sequence in parallel, we allow a word at position $i$
to receive information from a later word -- say at posiiton $j$, $i < j$.

For the language modeling application we are building, we want the context vector of a particular input word 
to not be influenced by words **following** it.

In 2D array language (where we compute all context vectors in parallel with matrix multiplications): Mask out all inputs  following position i in row i (all cells above the diagonal).

Recomputing:

In [49]:
queries = sa_v2.W_query(inputs)     #1
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [50]:
attn_weights.sum(axis=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

Here's the mask:

In [51]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


Now we apply it:

In [52]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


We've lost normalization:

In [53]:
masked_simple.sum(axis=1)

tensor([0.1921, 0.3700, 0.5357, 0.6775, 0.8415, 1.0000],
       grad_fn=<SumBackward1>)

In [54]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
# Renormalize
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [55]:
masked_simple_norm.sum(axis=1)

tensor([1., 1., 1., 1., 1., 1.], grad_fn=<SumBackward1>)

Same answer as above, but more efficient masking by using `-inf` before softmaxing:

In [57]:
# We'll fill attn scores where this is True with the mask value:
mask_simple.bool()

tensor([[ True, False, False, False, False, False],
        [ True,  True, False, False, False, False],
        [ True,  True,  True, False, False, False],
        [ True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True]])

In [58]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# Fill in the masked values with -inf
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)
# Now do the softmax
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


##  Dropout

Demonstrating dropout of .5 (non dropout vals are scaled up (* .5^{-1}) in order for the overall effect of attention module outputs on other modules to remain roughly constant.

In [59]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)    #1
example = torch.ones(6, 6)      #2
print(example)
print(dropout(example))

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [601]:
dropout = torch.nn.Dropout(0.5)
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


##  Batches (inputs are 3D arrays)

The learner needs to process input in **batches**.  The size of a batch is a hyperparameter and its optimal value is dependent in part on factors like the length of the sequences being processed and the RAM of the computer.
Learning (back propagation) happens after each batch so batch size shouldnot be too small.

To do this we are just going to stack the kinds ofinputs we've been discussing on top of each 
other. This will give us 3D arrays. Each **layer** of the 3D input is a single input sequence.

An example input with a batch size of 2.  The second and third dimension are the history length (also: sequence length or context length) and embedding dimension.

One context sequence:

In [61]:
print(inputs.shape)
inputs

torch.Size([6, 3])


tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [78]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  

torch.Size([2, 6, 3])


## Torch class with causal attention, dropout, and batches

In [83]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)            # New: dropout "layer"
        ##  Not a parameter but needs to be moved along with parameters
        ##  when we move data to GPU
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )             # New: call to `register_buffer` for upper tri mask

    def forward(self, x):
        b, context_length, d_in = x.shape    
        # keys is b x context_length x d_out
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # keys.transpose(1, 2)  is b x d_out x context_length
        # to be compatuible with queries matrix multpiplication mapping:
        # b x context_length x d_out
        # attn_scores is b x d_out x d_out
        attn_scores = queries @ keys.transpose(1, 2)   
        # PyTorch convention: operations with a trailing 
        # underscore are in-place.
        attn_scores.masked_fill_(                                                       
            self.mask.bool()[:context_length, :context_length], -torch.inf) 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec




In [84]:
batch.shape

torch.Size([2, 6, 3])

In [85]:
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

An input is a batch_size x seq_length batch.  The output is the corresponding batch_size X seq_lenth x d_out context vectors.  

In [86]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


## Multihead attention (pass 1)

In [87]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        # This loop (inside a list comprehension) will
        # be eliminated in the next more efficent version
        self.heads = nn.ModuleList(
            [CausalAttention(
                 d_in, d_out, context_length, dropout, qkv_bias
             ) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        # This loop too will have to go.
        return torch.cat([head(x) for head in self.heads], dim=-1)


Note  the actual dimension of the output vectors is twice `d_out`,
because we are using two attention heads and concatenating
the results.

In [88]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens ina context
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
print(batch.shape)
context_vecs = mha(batch)

print(context_vecs)
# batch size x context length x (num_heads * d_out) 
print("context_vecs.shape:", context_vecs.shape)


torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


#### EXERCISE 3.2 RETURNING TWO-DIMENSIONAL EMBEDDING VECTORS
Change the input arguments for the MultiHeadAttentionWrapper(..., num_ heads=2) call such that the output context vectors are two-dimensional instead of four-dimensional while keeping the setting num_heads=2. Hint: You don’t have to modify the class implementation; you just have to change one of the other input arguments.

#### Solution

In [73]:
# Change the doutargument from 2 to 1

mha2 = MultiHeadAttentionWrapper(
    d_in, 1, context_length, 0.0, num_heads=2
)
context_vecs = mha2(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)


tensor([[[0.7128, 0.4106],
         [0.8309, 0.3569],
         [0.8696, 0.3342],
         [0.7802, 0.2922],
         [0.7388, 0.2238],
         [0.7163, 0.2381]],

        [[0.7128, 0.4106],
         [0.8309, 0.3569],
         [0.8696, 0.3342],
         [0.7802, 0.2922],
         [0.7388, 0.2238],
         [0.7163, 0.2381]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


###  A  more efficient approach to multihead attention

#### Elementwise arithmetic review (illustrating a 3D case)

In [89]:
import numpy as np

M = np.arange(24).reshape((3,4,2))
M

array([[[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7]],

       [[ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15]],

       [[16, 17],
        [18, 19],
        [20, 21],
        [22, 23]]])

In [77]:
# N is another array of the same shape
O = np.ones((3, 4,2))
O[:,:,1] = 2

In [672]:
O

array([[[1., 2.],
        [1., 2.],
        [1., 2.],
        [1., 2.]],

       [[1., 2.],
        [1., 2.],
        [1., 2.],
        [1., 2.]],

       [[1., 2.],
        [1., 2.],
        [1., 2.],
        [1., 2.]]])

In [673]:
# Elementwise multiplication:
M*O

array([[[ 0.,  2.],
        [ 2.,  6.],
        [ 4., 10.],
        [ 6., 14.]],

       [[ 8., 18.],
        [10., 22.],
        [12., 26.],
        [14., 30.]],

       [[16., 34.],
        [18., 38.],
        [20., 42.],
        [22., 46.]]])

Now the same idea teleported to matrix multiplication, also a binary operation.  It just happens
to be defined for 2D matrices.

The rules of matrix multiplication impose the following shape requirements

```python
(m x n) @ (n x o) => (m,o)
```

Consider `M` as defined above, a 3 x 4 x 2 array.

Now think of `M` as an array containing 3 2D arrays

In [666]:
M[0].shape, M[1].shape, M[2].shape

((4, 2), (4, 2), (4, 2))

For elementwise compatibility  for matrix multiplication, we need an array containing 3 (2 x 4) arrays:

In [90]:
N = np.ones((3, 2, 4))
N[0].shape, N[1].shape, N[2].shape

((2, 4), (2, 4), (2, 4))

 The output of matrix mutiplication will be an array containing 3 4x4 arrays or a 3 x 4 x 4 array:
 
 ```python
(4 x 2) @ (2 x 4) => (4,4)
```

And it works:

In [91]:
print((M@N).shape)
M@N

(3, 4, 4)


array([[[ 1.,  1.,  1.,  1.],
        [ 5.,  5.,  5.,  5.],
        [ 9.,  9.,  9.,  9.],
        [13., 13., 13., 13.]],

       [[17., 17., 17., 17.],
        [21., 21., 21., 21.],
        [25., 25., 25., 25.],
        [29., 29., 29., 29.]],

       [[33., 33., 33., 33.],
        [37., 37., 37., 37.],
        [41., 41., 41., 41.],
        [45., 45., 45., 45.]]])

#### The 4D case (needed for multihead attention)

In [92]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],    
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

In [95]:
print(a.shape)
# a convenient way to get an array of compatible shape
print(a.transpose(2, 3).shape)

torch.Size([1, 2, 3, 4])
torch.Size([1, 2, 4, 3])


The last 2 dimensions are compatible for ordinary Matrix Multiplication (M). 
The shape mapping is:

```python
(3,4)@(4,3) -> (3,3)
```

Each of the 4D arrays has only one batch. 
There are two (3,4) matrices contained in `a[0]` and two (4,3) matrices
contained in `a.transpose[0]`, so we only have
two pairings to do.

The complete shape mapping is 

```python
(1,2,3,4)@(1,2,4,3) -> (1,2,3,3)
```

First the full matrix multiplication:

In [96]:
print((a@a.transpose(2, 3)).shape)
R = a@a.transpose(2, 3)
R

torch.Size([1, 2, 3, 3])


tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])

The takeaway point:  We are **not** extending the definition of matrix multiplication.
The relevant inner linear mpping is still represented by a 2D matrix.  Let's find the first pairing.

In [652]:
# 3 x 4
a[0, 0,:,:]

tensor([[0.2745, 0.6584, 0.2775, 0.8573],
        [0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340]])

In [653]:
# 4 x 3
a.transpose(2, 3)[0,0,:,:]

tensor([[0.2745, 0.8993, 0.7179],
        [0.6584, 0.0390, 0.7058],
        [0.2775, 0.9268, 0.9156],
        [0.8573, 0.7388, 0.4340]])

This gives the first of the layers in `R[0]`, a 3x3 matrix:

In [654]:
a[0, 0,:,:]@a.transpose(2, 3)[0,0,:,:]

tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

And here's the second layer:

In [655]:
a[0, 1,:,:]@a.transpose(2, 3)[0,1,:,:]

tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])

## Now add in broadcasting

`a[0, 0,:,:]` matches the last two dimensions of `a.transpose(2, 3)`.  So broadcast it:

In [98]:
xx = np.arange(24).reshape((8,3))
y = np.array([1,2,3])
print(xx.shape,xx)
print(y.shape,y)
xx * y

(8, 3) [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]
 [15 16 17]
 [18 19 20]
 [21 22 23]]
(3,) [1 2 3]


array([[ 0,  2,  6],
       [ 3,  8, 15],
       [ 6, 14, 24],
       [ 9, 20, 33],
       [12, 26, 42],
       [15, 32, 51],
       [18, 38, 60],
       [21, 44, 69]])

In [686]:
print(a[0, 0,:,:].shape)
print(a.transpose(2, 3).shape)
a[0, 0,:,:]@a.transpose(2, 3)

torch.Size([3, 4])
torch.Size([1, 2, 4, 3])


tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.7540, 1.2251, 1.0792],
          [0.6143, 1.5153, 1.2529],
          [0.6738, 1.2942, 1.3323]]]])

The same shape is output as in the example above, but `a[0,0,:,:]` has been used in both multiplications.  We rely on this kind of broadcasting quite a but in the learner code, because we apply the same 2D linear map  to each
of the (batch_size * context_length)-many word vectors in the input (for example, when applying
$W_{Q}$, $W_{K}$, and $W_{V}$ to the input vectors.

Compare what we got above when we use elementwise matrix multiplication instead of broadcasting.

In [687]:
a@a.transpose(2, 3)

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])

The second  3 x 4 matrix was produced by multiplication with `a[0,1,:,:]`.

##  Multihead attention module (efficient version)

In [47]:
class MultiHeadAttention(nn.Module):
    """
    x_out = self.forward(x_in)
    
    x_in: batch_size x context_length x d_in 
    
    x_out: batch_size x context_length x d_out
    
    Note that an instantiated learner is indifferent to 
    the particular value of batch_size. The assumption is
    simply that the input is a 3D matrix and the first dimension is batch
    size, but only context_length, d_in, and d_out are parameters of 
    the learner.
    
    The d_out dimensions will be evenly divided among the heads, so
    d_out must be evenly divisible by num_heads. 
    """
    def __init__(self, d_in, d_out, 
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Will divide the d_out dimensions evenly among the heads
        self.head_dim = d_out // num_heads    #1
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)   
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, context_length, d_in = x.shape
        # keys is now b x context_length x d_out
        keys = self.W_key(x)         #3
        queries = self.W_query(x)    #3
        values = self.W_value(x)     #3

        # dividing d_out into num_heads components; each component has shape head_dim 
        # keys is now b x context_length x num_heads x head_dim
        keys = keys.view(b, context_length, self.num_heads, self.head_dim)       #4
        values = values.view(b, context_length, self.num_heads, self.head_dim)  
        queries = queries.view(                                             
            b, context_length, self.num_heads, self.head_dim                    
        )                                                                   


        # keys becomes b x num_heads x context_length x head_dim
        keys = keys.transpose(1, 2)          #5
        queries = queries.transpose(1, 2)    #5
        values = values.transpose(1, 2)      #5

        # make keys compatible for matrix multiplication 
        #   mapping     @  mapped
        #   (..., m, n) @ (..., n, m)
        # keys becomes b x num_heads x head_dim X context_length
        # attn_scores will be b x num_heads x context_length x context_length
        attn_scores = queries @ keys.transpose(2, 3)   #6
        mask_bool = self.mask.bool()[:context_length, :context_length]    #7

        attn_scores.masked_fill_(mask_bool, -torch.inf)     #8

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # attn_weights : b x num_heads x context_length x context_length
        # values :  b x num_heads x context_length x head_dim
        # context_vec:  b x num_heads x context_length x head_dim
        # transpose(1, 2) =>
        # context_vec:  b x context_length x num_heads x head_dim
        context_vec = (attn_weights @ values).transpose(1, 2)   #9
        # Recombine num_heads x head_dim into d_out
        # context_vec:  b x context_length x d_out
        context_vec = context_vec.contiguous().view(
            b, context_length, self.d_out
        )                                           #10
        # And one more linear map for good luck to produce what we might call logits;
        # 2D map is a square matrix:
        # context_vec:  b x context_length x d_out
        context_vec = self.out_proj(context_vec)    #11
        return context_vec       

##  Output layers, evaluation, learning loop

This material is from Chapters 4 and 5 and concludes the building of a working LLM.

The elements follow the publicly released code for GPT-2 but we obviously 
will work at a smaller scale here.  

In [ ]:
!

In [166]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size.  Weve already used this vocab in teh GPOT tolkenzier
    "context_length": 1024,  # Context length.  Hoo boy!  Bug step up
    "emb_dim": 768,          # Embedding dimension    More than doubling the embedding dimensionality above
    "n_heads": 12,           # Number of attention heads.  More memory
    "n_layers": 12,          # Number of layers.  More meory still!
    "drop_rate": 0.1,        # Dropout rate  modest
    "qkv_bias": False        # Query-Key-Value bias. for later
    "loss_type": 
}

gpt = GPT_CONFIG_124M

In [46]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # a sequence of n_layers transformer blocks.
        self.trf_blocks = nn.Sequential(               
            *[DummyTransformerBlock(cfg)               
              for _ in range(cfg["n_layers"])]         
        )                                             
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])    
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        # The inout is a batch_size x seq_len array
        # of vocab indices (what is put out by our tokenizer)
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        # position embeddings:just use the embeddings of the first seq_length vocab items
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds 
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class DummyTransformerBlock(nn.Module):    
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):     #4
        return x

class DummyLayerNorm(nn.Module):           
    def __init__(self, normalized_shape, eps=1e-5):   
        super().__init__()

    def forward(self, x):
        return x

In [103]:
txt1_tokens

[6109, 3626, 6100, 345]

In [53]:
#txts = "Every effort moves you<|endoftext|>Every day holds a"
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

txt1_tokens = tokenizer.encode(txt1,allowed_special={"<|endoftext|>"})
batch.append(torch.tensor(txt1_tokens))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


Output logits assign a single floating point "score" to each vocab item for each item in context
for each item in batch.

In [125]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [126]:
logits.dtype

torch.float32

Memory check; nbytes= Array_size * byte_number_per_item

In [146]:
logits.nbytes/(2*4*50257)

4.0

And the embeddings in the model take up way more space, since we need one embeeding vector per vector item:

In [147]:
#dir(model)
t_emb = model.tok_emb
print(t_emb.weight.shape)#.nbytes
t_emb.weight.nbytes/(50257 * 768)

torch.Size([50257, 768])


4.0

## Normalizing layer

Change embedding values to have mean of 0 and variance of 1.

Illustrate with some random data passed through a linear layer and made non-negative by 
applying `ReLU`.

In [52]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)     #1
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


Compute mean and variance of `out`.

In [190]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


Apply the normalizing transformation: subtract $mean$ from all values in `out`, divide all results by 
the $standard\, deviation$, or $\sqrt{variance}$.

Compute mean and variance of result to verify:

In [160]:
torch.set_printoptions(sci_mode=False)
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)

In [161]:
print("Normalized layer outputs:\n", out_norm)
print("Mean:\n", mean)
print("Variance:\n", var)

Normalized layer outputs:
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[    -0.0000],
        [     0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [45]:
class LayerNorm(nn.Module):
    """
    
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

> The scale and shift are two trainable parameters (of the same dimension as the input) that the LLM automatically adjusts during training if it is determined that doing so would improve the model’s performance on its training task. This allows the model to learn appropriate scaling and shifting that best suit the data it is processing. -- Raschka 4.2

In [936]:
emb_dim = out.shape[1]
ln = LayerNorm(emb_dim=emb_dim)
out_ln = ln(out)

# Verify
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[     0.0000],
        [    -0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.9995],
        [0.9997]], grad_fn=<VarBackward0>)


### Adding a GELU layer

To see the close relationship of GELU and RELU, see GELU plot (Fig. 4.8).  Essentially
GELU can be thought of as a smoothed version of RELU, much as softmax can be thought of
a smoothed version of argmax or max.  In the example below

In [44]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

# We wrap it in a feed forward layer
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [167]:

ffn = FeedForward(cfg)
x = torch.rand(2, 3, 768)          #1
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


In [168]:
cfg = GPT_CONFIG_124M
print("x ",x.shape)
x1 = nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"])(x)
print("x1",x1.shape,x1.min())
x2 = GELU()(x1)
print("x2",x2.shape,x2.min())
x3 = nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])(x2)
print("x3", x3.shape,x3.min())

x  torch.Size([2, 3, 768])
x1 torch.Size([2, 3, 3072]) tensor(-1.3397, grad_fn=<MinBackward1>)
x2 torch.Size([2, 3, 3072]) tensor(-0.1700, grad_fn=<MinBackward1>)
x3 torch.Size([2, 3, 768]) tensor(-0.4094, grad_fn=<MinBackward1>)


### Brief discussion of shortcuts

**Shortcut** connections are sometimes called **skip** connections.  Their chief
function is to address the **vanishing gradient problem**.

In deep networks, gradients are multiplied across layers during backpropagation, causing them to shrink to near zero. A residual block adds the input ($x$) directly to the output of a layer ($F(x)$), so that
the output of the layer is $x + F(x)$.  Therefore the order of magnitude of the output is maintained.

###  A Transformer Block

In [42]:
class TransformerBlock(nn.Module):
    """
    Required input argument is a GPT 
    conifiguration dictionary (GPT_CONFIG_124M in this NB).
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x  #1
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #2

        shortcut = x         #3
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #4
        return x
#1 Shortcut connection for attention block
#2 Add the original input back
#3 Shortcut connection for feed forward block
#4 Adds the original input back

In [51]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768)                   #1
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

#Input shape: torch.Size([2, 4, 768])
#Output shape: torch.Size([2, 4, 768])
print("Input shape:", x.shape)
print("Output shape:", output.shape)
#1 Creates sample input of shape [batch_size, num_tokens, emb_dim]

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


As we can see, the transformer block maintains the input dimensions in its output, indicating that the transformer architecture processes sequences of data without altering their shape throughout the network.

The preservation of shape throughout the transformer block architecture is not incidental but a crucial aspect of its design. This design enables its effective application across a wide range of sequence-to-sequence tasks, where each output vector directly corresponds to an input vector, maintaining a one-to-one relationship. However, the output is a context vector that encapsulates information from the entire input sequence (see chapter 3). This means that while the physical dimensions of the sequence (length and feature size) remain unchanged as it passes through the transformer block, the content of each output vector is re-encoded to integrate contextual information from across the entire input sequence.

With the transformer block implemented, we now have all the building blocks needed to implement the GPT architecture. As illustrated in figure 4.14, the transformer block combines layer normalization, the feed forward network, GELU activations, and shortcut connections. As we will eventually see, this transformer block will make up the main component of the GPT architecture.

figure
Figure 4.14 The building blocks necessary to build the GPT architecture. The black checks indicate the blocks we have completed.
4.6 Coding the GPT model
We started this chapter with a big-picture overview of a GPT architecture that we called DummyGPTModel. In this DummyGPTModel code implementation, we showed the input and outputs to the GPT model, but its building blocks remained a black box using a DummyTransformerBlock and DummyLayerNorm class as placeholders.

### The GPT model

1. Replace the DummyTransformerBlock and DummyLayerNorm placeholders with the real TransformerBlock and LayerNorm classes we coded previously.
2. This isa fully working version of the original 124-million-parameter version of GPT-2. 
3. In chapter 5, Raschke pretrains a GPT-2 model.
4. In chapter 6, he loads in the pretrained weights from OpenAI.

Figure 4.15  => GPT model architecture.png

The transformer block is repeated 12 times (see the GPT_CONFIG_124M dictionary). This transform block is repeated 48 times in the largest GPT-2 model with 1,542 million parameters.

#### Figure 4.15  => GPT model architecture.png

Let’s now code the architecture in figure 4.15.

In [49]:
#Listing 4.7 The GPT model architecture implementation

import torch
from torch import nn

GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size.  Weve already used this vocab in teh GPOT tolkenzier
    "context_length": 1024,  # Context length.  Hoo boy!  Bug step up
    "emb_dim": 768,          # Embedding dimension    More than doubling the embedding dimensionality above
    "n_heads": 12,           # Number of attention heads.  More memory
    "n_layers": 12,          # Number of layers.  More meory still!
    "drop_rate": 0.1,        # Dropout rate  modest
    "qkv_bias": False        # Query-Key-Value bias. for later
    #"loss_type": 
}

gpt = GPT_CONFIG_124M

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
 #1
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
    
#1 The device setting will allow us to train 
#  the model on a CPU or GPU, depending on which device the input data sits on.

Thanks to the TransformerBlock class, the GPTModel class is relatively small and compact.

The __init__ constructor of this GPTModel class initializes the token and positional embedding layers using the configurations passed in via a Python dictionary, cfg. These embedding layers are responsible for converting input token indices into dense vectors and adding positional information (see chapter 2).

Next, the __init__ method creates a sequential stack of TransformerBlock modules equal to the number of layers specified in cfg. Following the transformer blocks, a LayerNorm layer is applied, standardizing the outputs from the transformer blocks to stabilize the learning process. Finally, a linear output head without bias is defined, which projects the transformer’s output into the vocabulary space of the tokenizer to generate logits for each token in the vocabulary.

The forward method takes a batch of input token indices, computes their embeddings, applies the positional embeddings, passes the sequence through the transformer blocks, normalizes the final output, and then computes the logits, representing the next token’s unnormalized probabilities. We will convert these logits into tokens and text outputs in the next section.

Let’s now initialize the 124-million-parameter GPT model using the GPT_CONFIG_ 124M dictionary we pass into the cfg parameter and feed it with the batch text input we previously created:

In [64]:
import inspect
inspect.signature(model.__call__)

<Signature (*args, **kwargs)>

In [60]:
#[att for att in dir(model) if "oss" in att]
dir(model)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',
 '_named_members',
 '_non_persistent_buffers_se

In [54]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
# 2 (batch sz) by 4 ()
print("Input batch:\n", batch)
# [2, 4, 50257] last is GPT vocab size
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


This code prints the contents of the input batch followed by the output tensor:

```python
Input batch:
 tensor([[6109,  3626,  6100,   345],      #1 Token IDs of text 1

         [6109,  1110,  6622,   257]])     #2 Token IDs of text 2
```

```python
Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)
```

As we can see, the output tensor has the shape [2, 4, 50257], since we passed in two input texts with four tokens each. The last dimension, 50257, corresponds to the vocabulary size of the tokenizer. Later, we will see how to convert each of these 50,257-dimensional output vectors back into tokens.

Before we move on to coding the function that converts the model outputs into text, let’s spend a bit more time with the model architecture itself and analyze its size. Using the numel() method, short for “number of elements,” we can collect the total number of parameters in the model’s parameter tensors:

In [175]:
#Total number of parameters: 163,009,536
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


Now, a curious reader might notice a discrepancy. Earlier, we spoke of initializing a 124-million-parameter GPT model, so why is the actual number of parameters 163 million?

The reason is a concept called weight tying, 
which was used in the original GPT-2 architecture. It means that the original GPT-2 architecture reuses the weights from the token embedding layer in its output layer. To understand better, let’s take a look at the shapes of the token embedding layer and linear output layer that we initialized on the model via the GPTModel earlier:

In [176]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


As we can see from the print outputs, the weight tensors for both these layers have the same shape:

```
Token embedding layer shape: torch.Size([50257, 768])
Output layer shape:          torch.Size([50257, 768])
```
The token embedding and output layers are very large due to the number of rows for the 50,257 in the tokenizer’s vocabulary. Let’s remove the output layer parameter count from the total GPT-2 model count according to the weight tying:

In [177]:
total_params_gpt2 = (
    total_params - sum(p.numel()
    for p in model.out_head.parameters())
)
#Number of trainable parameters considering weight tying: 124,412,160
print(f"Number of trainable parameters "
      f"considering weight tying: {total_params_gpt2:,}"
)

Number of trainable parameters considering weight tying: 124,412,160



As we can see, the model is now only 124 million parameters large, matching the original size of the GPT-2 model.

**Weight tying** 
reduces the overall memory footprint and computational complexity of the model. But it can affect
performance.

**Exercise 4.1** Number of parameters in feed forward and attention modules
Calculate and compare the number of parameters that are contained in the feed forward module and those that are contained in the multi-head attention module.

In [178]:
total_size_bytes = total_params * 4       #1
total_size_mb = total_size_bytes / (1024 * 1024)     #2
# Total size of the model: 621.83 MB
print(f"Total size of the model: {total_size_mb:.2f} MB")
#1 Calculates the total size in bytes (assuming float32, 4 bytes per parameter)
#2 Converts to megabytes

Total size of the model: 621.83 MB


In conclusion, by calculating the memory requirements for the 163 million parameters in our GPTModel object and assuming each parameter is a 32-bit float taking up 4 bytes, we find that the total size of the model amounts to 621.83 MB, illustrating the relatively large storage capacity required to accommodate even relatively small LLMs.

Now that we’ve implemented the GPTModel architecture and saw that it outputs numeric tensors of shape [batch_size, num_tokens, vocab_size], let’s write the code to convert these output tensors into text.

Exercise 4.2 Initializing larger GPT models
We initialized a 124-million-parameter GPT model, which is known as “GPT-2 small.” Without making any code modifications besides updating the configuration file, use the GPTModel class to implement GPT-2 medium (using 1,024-dimensional embeddings, 24 transformer blocks, 16 multi-head attention heads), GPT-2 large (1,280-dimensional embeddings, 36 transformer blocks, 20 multi-head attention heads), and GPT-2 XL (1,600-dimensional embeddings, 48 transformer blocks, 25 multi-head attention heads). As a bonus, calculate the total number of parameters in each GPT model.

### Generating text

We will now implement the code that converts the tensor outputs of the GPT model back into text. Before we get started, let’s briefly review how a generative model like an LLM generates text one word (or token) at a time.

figure
Figure 4.16 The step-by-step process by which an LLM generates text, one token at a time. Starting with an initial input context (“Hello, I am”), the model predicts a subsequent token during each iteration, appending it to the input context for the next round of prediction. As shown, the first iteration adds “a,” the second “model,” and the third “ready,” progressively building the sentence.
Figure 4.16 illustrates the step-by-step process by which a GPT model generates text given an input context, such as “Hello, I am.” With each iteration, the input context grows, allowing the model to generate coherent and contextually appropriate text. By the sixth iteration, the model has constructed a complete sentence: “Hello, I am a model ready to help.” We’ve seen that our current GPTModel implementation outputs tensors with shape [batch_size, num_token, vocab_size]. Now the question is: How does a GPT model go from these output tensors to the generated text?

The process by which a GPT model goes from output tensors to generated text involves several steps, as illustrated in figure 4.17. These steps include decoding the output tensors, selecting tokens based on a probability distribution, and converting these tokens into human-readable text.

figure
Figure 4.17 The mechanics of text generation in a GPT model by showing a single iteration in the token generation process. The process begins by encoding the input text into token IDs, which are then fed into the GPT model. The outputs of the model are then converted back into text and appended to the original input text.
The next-token generation process detailed in figure 4.17 illustrates a single step where the GPT model generates the next token given its input. In each step, the model outputs a matrix with vectors representing potential next tokens. The vector corresponding to the next token is extracted and converted into a probability distribution via the softmax function. Within the vector containing the resulting probability scores, the index of the highest value is located, which translates to the token ID. This token ID is then decoded back into text, producing the next token in the sequence. Finally, this token is appended to the previous inputs, forming a new input sequence for the subsequent iteration. This step-by-step process enables the model to generate text sequentially, building coherent phrases and sentences from the initial input context.

In practice, we repeat this process over many iterations, such as shown in figure 4.16, until we reach a user-specified number of generated tokens. In code, we can implement the token-generation process as shown in the following listing.

In [191]:
#Listing 4.8 A function for the GPT model to generate text
def generate_text_simple(model, idx,                 #1
                         max_new_tokens, context_size): 
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]    #2
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]                    #3
        probas = torch.softmax(logits, dim=-1)           #4
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)    #5
        idx = torch.cat((idx, idx_next), dim=1)     #6

    return idx
#1 idx is a (batch, n_tokens) array of indices in the current context.
#2 Crops current context if it exceeds the supported context size, e.g., if LLM supports only 5 tokens, and the context size is 10, then only the last 5 tokens are used as context
#3 Focuses only on the last time step, so that (batch, n_token, vocab_size) becomes (batch, vocab_size)
#4 probas has shape (batch, vocab_size).
#5 idx_next has shape (batch, 1).
#6 Appends sampled index to the running sequence, where idx has shape (batch, n_tokens+1)

A generative loop for a language model using PyTorch. 

It iterates for a specified number of new tokens to be generated, crops the current context to fit the model’s maximum context size, computes predictions, and then selects the next token based on the highest probability prediction.

To code the generate_text_simple function, we use a softmax function to convert the logits into a probability distribution from which we identify the position with the highest value via torch.argmax. The softmax function is monotonic, meaning it preserves the order of its inputs when transformed into outputs. So, in practice, the softmax step is redundant since the position with the highest score in the softmax output tensor is the same position in the logit tensor. In other words, we could apply the torch.argmax function to the logits tensor directly and get identical results. However, I provide the code for the conversion to illustrate the full process of transforming logits to probabilities, which can add additional intuition so that the model generates the most likely next token, which is known as greedy decoding.

In [192]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)    #1
#encoded: [15496, 11, 314, 716]
#encoded_tensor.shape: torch.Size([1, 4])
print("encoded_tensor.shape:", encoded_tensor.shape)
#1 Adds batch dimension

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


Next, we put the model into .eval() mode. 
This disables random components like dropout, 
which are only used during training, and use the generate_text_simple function on the encoded input tensor:

In [193]:
model.eval()                  #1
out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

#Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843,
#30961, 42348,  7267]])
print("Output:", out)
#Output length: 10
print("Output length:", len(out[0]))
#1 Disables dropout since we are not training the model

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


Using the .decode method of the tokenizer, we can convert the IDs back into text:

In [194]:
#Hello, I am Featureiman Byeswickattribute argue
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
# Gibberish
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue


The mopdel isuntrained.  Model training is the topic of chapter 5.

####  Exercise 4.3 
Using separate dropout parameters
At the beginning of this chapter, we defined a global drop_rate setting in the GPT_ CONFIG_124M dictionary to set the dropout rate in various places throughout the GPTModel architecture. Change the code to specify a separate dropout value for the various dropout layers throughout the model architecture. (Hint: there are three distinct places where we used dropout layers: the embedding layer, shortcut layer, and multi-head attention module.)

### Summary

1. Layer normalization stabilizes training by ensuring that each layer’s outputs have a consistent mean and variance.
2. Shortcut connections are connections that skip one or more layers by feeding the output of one layer directly to a deeper layer, which helps mitigate the vanishing gradient problem when training deep neural networks, such as LLMs.
3/ Transformer blocks are a core structural component of GPT models, combining masked multi-head attention modules with fully connected feed forward networks that use the GELU activation function.
4. GPT models are LLMs with many repeated transformer blocks that have millions to billions of parameters.
5. GPT models come in various sizes, for example, 124, 345, 762, and 1,542 million parameters, which we can implement with the same GPTModel Python class.
6. The text-generation capability of a GPT-like LLM involves decoding output tensors into human-readable text by sequentially predicting one token at a time based on a given input context.
7. Without training, a GPT model generates incoherent text.

## Perplexity using gpt-2 pretrained models


I have created **my own environment** for this part of the notebook.  It is called `transformers`. I did all the installs required below into this environment, You are advised to do something  similar if you are running on your home machine.

This is note to myself to run this part of the notebook only in that environment.


```
conda create --name esm python=3.10
 conda activate esm
 conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
 pip3 install -r requirements.txt
```

The contents of `requirements.txt`:

```
accelerate
datasets==2.20.0
pyfastx
transformers
boto3
huggingface_hub==0.23.4
```

And then I get:


Hugging Face route.  Installs 


```
conda install conda-forge::transformers
conda install conda-forge::datasets
conda install -c conda-forge pyarrow
```


The point of the `accelerate` module is to manage moving data and models to the GPU/TPU/MPS on your machine
when its feasible and efficient, as it usually is with HuggingFace models.

Ran `accelerate config` and then checked state with `accelerate env`:

Output from `accelerate env` (this environment worked):

```
- `Accelerate` version: 1.13.0
- Platform: macOS-26.3.1-arm64-arm-64bit
- `accelerate` bash location: /Users/gawron/opt/anaconda3/envs/transformers/bin/accelerate
- Python version: 3.10.14
- Numpy version: 2.2.6
- PyTorch version: 2.10.0
- PyTorch accelerator: N/A
- System RAM: 16.00 GB
- `Accelerate` default config:
	- compute_environment: LOCAL_MACHINE
	- distributed_type: MULTI_MLU
	- mixed_precision: no
	- use_cpu: False
	- debug: False
	- num_processes: 1
	- machine_rank: 0
	- num_machines: 1
	- rdzv_backend: static
	- same_network: True
	- main_training_function: main
	- enable_cpu_affinity: False
	- downcast_bf16: no
	- tpu_use_cluster: False
	- tpu_use_sudo: False
	- tpu_env: []
```

The environment in which I was unable to get this part of the notebook working:

```
- Accelerate` version: 1.13.0
- Platform: macOS-26.3-arm64-arm-64bit
- `accelerate` bash location: /Users/gawron/opt/anaconda3/envs/p312/bin/accelerate
- Python version: 3.12.5
- Numpy version: 2.4.2
- PyTorch version: 2.5.1
- PyTorch accelerator: N/A
- System RAM: 16.00 GB
- `Accelerate` default config:
	- compute_environment: LOCAL_MACHINE
	- distributed_type: MULTI_MLU
	- mixed_precision: no
	- use_cpu: False
	- debug: False
	- num_processes: 1
	- machine_rank: 0
	- num_machines: 1
	- rdzv_backend: static
	- same_network: True
	- main_training_function: main
	- enable_cpu_affinity: False
	- downcast_bf16: no
	- tpu_use_cluster: False

```

In [19]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from accelerate import Accelerator

device = Accelerator().device

In [20]:
device

device(type='mps')

Go to 

https://huggingface.co/

Setup an account if you don't already have one.  Go to settings and Choose "access_tokens" from the menu on the left. Generate a token with permission "finegrained". Save it in a file on your machine somewhere.  Don't share.  

The code seems to work without access tokens.  But you will get warnings about unauthorized access.

In [21]:
import os
HF_TOKEN = os.environ["HF_TOKEN"]
model_id = "openai-community/gpt2-large"
model = GPT2LMHeadModel.from_pretrained(model_id,token=HF_TOKEN).to(device)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

In [6]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_id,token=HF_TOKEN)

The next cell is optional, but it gives you a chance to type in your access token.

In [1]:
#from huggingface_hub import notebook_login

#notebook_login()

In [7]:
from datasets import load_dataset

test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test",token=HF_TOKEN)

This warning is ok.  The data is appropriately chunked by the `stride` parameter in the following cell.

```
Token indices sequence length is longer than the specified maximum sequence length for this model (287644 > 1024). Running this sequence through the model will result in indexing errors
```

In [31]:
encodings = tokenizer("\n\n".join(test["text"]), return_tensors="pt")

Token indices sequence length is longer than the specified maximum sequence length for this model (287644 > 1024). Running this sequence through the model will result in indexing errors


In [35]:
import torch
from tqdm import tqdm

max_length = model.config.n_positions
stride = 512
seq_len = encodings.input_ids.size(1)

nll_sum = 0.0
n_tokens = 0
prev_end_loc = 0
for begin_loc in tqdm(range(0, seq_len, stride)):
    end_loc = min(begin_loc + max_length, seq_len)
    trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
    input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
    target_ids = input_ids.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

        # loss is calculated using CrossEntropyLoss which averages over valid labels
        # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
        # to the left by 1.
        neg_log_likelihood = outputs.loss

    # Accumulate the total negative log-likelihood and the total number of tokens
    num_valid_tokens = (target_ids != -100).sum().item()  # number of valid tokens in target_ids
    batch_size = target_ids.size(0)
    num_loss_tokens = num_valid_tokens - batch_size  # subtract batch_size due to internal label shift
    nll_sum += neg_log_likelihood * num_loss_tokens
    n_tokens += num_loss_tokens
    prev_end_loc = end_loc
    if end_loc == seq_len:
        break

avg_nll = nll_sum / n_tokens  # average negative log-likelihood per token
ppl = torch.exp(avg_nll)

100%|████████████████████████████████████████▊| 560/562 [09:12<00:01,  1.01it/s]


And the final perplexity is:

In [36]:
ppl

tensor(16.4443, device='mps:0')

From [Hugging Face Perplexity Tutorial](https://huggingface.co/docs/transformers/perplexity)

>Running this with the stride length equal to the max input length is equivalent to the suboptimal, non-sliding-window strategy we discussed above. The smaller the stride, the more context the model will have in making each prediction, and the better the reported perplexity will typically be.

>When we run the above with stride = 1024, i.e. no overlap, the resulting PPL is 19.44, which is about the same as the 19.93 reported in the GPT-2 paper. By using stride = 512 and thereby employing our striding window strategy, this jumps down to 16.44. This is not only a more favorable score, but is calculated in a way that is closer to the true autoregressive decomposition of a sequence likelihood.

